# 03. Panel base (focos de calor + contexto DETER)

**Propósito.** Construir el **panel completo** `(municipio_norm, uf)` por semana: BDQueimadas es el esqueleto (con ceros reales cuando no hubo focos) y DETER aporta solo contexto para un KPI. Aquí no se construyen rezagos, targets, recortes, imputación ni división de datos: eso se hace después en Data Preparation.

**Entradas:** `BDQueimadas/bdqueimadas_limpio.csv`, `BDQueimadas/centroides_municipio.csv`, `DETER/deter_limpio.csv`. Opcional (comparación informativa): `datos/datasets_eda/dataset_semanal_COMPLETO.csv`.

**Salida:** `datos/datasets_preprocesamiento/panel_semanal_base.csv`.

**Qué cambió respecto al Hito 2**
- Se reescribió por completo: ya no existe el dataset a nivel de alerta ni las ventanas móviles de 7 días por alerta.
- El panel ya no es "las semanas con alertas DETER": es la grilla completa municipio por semana (251 semanas, de 2022-01-03 a 2026-10-19).
- Los centroides salen de los focos (no de DETER) y DETER solo suma alertas y hectáreas como contexto.
- Una semana de un municipio sin focos tiene `num_incendios = 0`, y las variables promedio quedan en NaN (no se imputa).

In [2]:
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1. Entorno: Google Colab (Drive) o ejecucion local
# ---------------------------------------------------------------------------
USAR_DRIVE = True   # True en Colab; False para trabajar con la carpeta local del repositorio
BASE_DRIVE = Path("/content/drive/MyDrive/BIODIVERSITY_GUARD")

if USAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = BASE_DRIVE
    except ImportError:
        print("No se detecto Google Colab: se usa la raiz local del repositorio.")
        USAR_DRIVE = False

if not USAR_DRIVE:
    # la raiz es la primera carpeta (hacia arriba desde la actual) que contiene "datos"
    _aqui = Path.cwd().resolve()
    BASE = next((p for p in [_aqui, *_aqui.parents] if (p / "datos").is_dir()), None)
    if BASE is None:
        raise FileNotFoundError("No se encontro la carpeta 'datos' subiendo desde " + str(_aqui))

if not BASE.exists():
    raise FileNotFoundError(f"No existe BASE = {BASE}. Verifica la carpeta BIODIVERSITY_GUARD en Drive.")

# ---------------------------------------------------------------------------
# 2. Rutas (todas salen de BASE)
# ---------------------------------------------------------------------------
DIR_PREP = BASE / "datos" / "datasets_preprocesamiento"
DIR_BDQ = DIR_PREP / "BDQueimadas"
DIR_DETER = DIR_PREP / "DETER"
DIR_NASA = DIR_PREP / "NasaPower"
DIR_SISAM = DIR_PREP / "SISAM"
DIR_EDA = BASE / "datos" / "datasets_eda"

# ---------------------------------------------------------------------------
# 3. Constantes comunes (identicas en los 7 notebooks)
# ---------------------------------------------------------------------------
FECHA_INICIO_DATOS = "2022-01-01"
FECHA_FIN_EXTRACCION = "2026-11-01"      # DETER y BDQueimadas
PRIMERA_SEMANA = "2022-01-03"             # primer lunes con semana completa (2022-01-01 es sabado)
ULTIMA_SEMANA = "2026-10-19"              # lunes de la ultima semana completa (termina domingo 2026-10-25)
SATELITE_REFERENCIA = "AQUA_M-T"
ESTADOS_AMAZONIA_LEGAL = ["ACRE", "AMAPÁ", "AMAZONAS", "MARANHÃO", "MATO GROSSO", "PARÁ", "RONDÔNIA", "RORAIMA", "TOCANTINS"]
MAPA_UF = {"ACRE": "AC", "AMAPÁ": "AP", "AMAZONAS": "AM", "MARANHÃO": "MA", "MATO GROSSO": "MT", "PARÁ": "PA", "RONDÔNIA": "RO", "RORAIMA": "RR", "TOCANTINS": "TO"}
CORRECCIONES_MUNICIPIO = {"ELDORADO DOS CARAJAS": "ELDORADO DO CARAJAS", "SANTA ISABEL DO PARA": "SANTA IZABEL DO PARA"}
FECHA_CORTE_CENTROIDES = "2024-12-31"     # los centroides se fijan con focos hasta esta fecha

# ---------------------------------------------------------------------------
# 4. Funciones comunes
# ---------------------------------------------------------------------------
def normalizar_texto(texto):
    # mayusculas, sin tildes ni espacios sobrantes
    if pd.isna(texto):
        return texto
    texto = str(texto).upper().strip()
    return unicodedata.normalize("NFKD", texto).encode("ASCII", "ignore").decode("utf-8")


def semana_lunes(fecha):
    # lunes de la semana (lunes a domingo) de cada fecha; recibe una Serie de fechas
    f = pd.to_datetime(fecha).dt.normalize()
    return f - pd.to_timedelta(f.dt.day_of_week, unit="D")


SEMANAS = pd.date_range(PRIMERA_SEMANA, ULTIMA_SEMANA, freq="W-MON")   # calendario de semanas completas
assert len(SEMANAS) == 251, f"El calendario deberia tener 251 semanas y tiene {len(SEMANAS)}"
assert (SEMANAS.day_of_week == 0).all(), "Todas las semanas deben empezar en lunes"


def verificar_entradas(archivos):
    # imprime los archivos de entrada con su tamano y falla si falta alguno
    faltan = []
    for a in map(Path, archivos):
        if a.exists():
            print(f"  OK  {a.relative_to(BASE)}  ({a.stat().st_size / 1e6:,.1f} MB)")
        else:
            faltan.append(a)
    if faltan:
        lista = "\n".join(f"  - {a}" for a in faltan)
        raise FileNotFoundError("Faltan archivos de entrada (copialos a esas carpetas y vuelve a ejecutar):\n" + lista)


print("BASE:", BASE)
print("Calendario:", SEMANAS[0].date(), "a", SEMANAS[-1].date(), f"({len(SEMANAS)} semanas)")

RUTA_FOCOS = DIR_BDQ / "bdqueimadas_limpio.csv"
RUTA_CENTROIDES = DIR_BDQ / "centroides_municipio.csv"
RUTA_DETER = DIR_DETER / "deter_limpio.csv"
RUTA_PANEL_HITO2 = DIR_EDA / "dataset_semanal_COMPLETO.csv"
RUTA_SALIDA = DIR_PREP / "panel_semanal_base.csv"
CLAVE = ["municipio_norm", "uf"]

print("\nArchivos de entrada:")
verificar_entradas([RUTA_FOCOS, RUTA_CENTROIDES, RUTA_DETER])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE: /content/drive/MyDrive/BIODIVERSITY_GUARD
Calendario: 2022-01-03 a 2026-10-19 (251 semanas)

Archivos de entrada:
  OK  datos/datasets_preprocesamiento/BDQueimadas/bdqueimadas_limpio.csv  (32.7 MB)
  OK  datos/datasets_preprocesamiento/BDQueimadas/centroides_municipio.csv  (0.0 MB)
  OK  datos/datasets_preprocesamiento/DETER/deter_limpio.csv  (38.4 MB)


## 1. Carga

Las correcciones de nomenclatura se aplican a DETER (`ELDORADO DOS CARAJAS` y `SANTA ISABEL DO PARA`) para que coincida con los nombres de BDQueimadas. La clave de cruce es siempre `(municipio_norm, uf)`.

In [3]:
focos = pd.read_csv(RUTA_FOCOS, parse_dates=["fecha"])
centroides = pd.read_csv(RUTA_CENTROIDES)
deter = pd.read_csv(RUTA_DETER, parse_dates=["view_date"])
deter["municipio_norm"] = deter["municipio_norm"].replace(CORRECCIONES_MUNICIPIO)

print("Focos:", focos.shape, "| rango:", focos["fecha"].min().date(), "a", focos["fecha"].max().date())
print("DETER:", deter.shape, "| rango:", deter["view_date"].min().date(), "a", deter["view_date"].max().date())
print("Centroides:", centroides.shape)

focos["semana"] = semana_lunes(focos["fecha"])
deter["semana"] = semana_lunes(deter["view_date"])

# solo las semanas completas del calendario entran al panel
focos_ok = focos[focos["semana"].isin(SEMANAS)]
deter_ok = deter[deter["semana"].isin(SEMANAS)]
descartados_focos = len(focos) - len(focos_ok)
descartados_deter = len(deter) - len(deter_ok)
print(f"\nFocos descartados por caer fuera del calendario: {descartados_focos:,} de {len(focos):,}")
print(f"Alertas DETER descartadas por caer fuera del calendario: {descartados_deter:,} de {len(deter):,}")

Focos: (445636, 11) | rango: 2022-01-01 a 2026-10-31
DETER: (221680, 18) | rango: 2022-01-01 a 2026-10-30
Centroides: (543, 6)

Focos descartados por caer fuera del calendario: 3,044 de 445,636
Alertas DETER descartadas por caer fuera del calendario: 601 de 221,680


## 2. Universo de municipios y grilla completa

El universo es la unión de los pares `(municipio_norm, uf)` presentes en focos y en DETER. La grilla es el producto cruzado de ese universo con las 251 semanas del calendario.

In [4]:
p_focos = focos[CLAVE].drop_duplicates()
p_deter = deter[CLAVE].drop_duplicates()
universo = pd.concat([p_focos, p_deter]).drop_duplicates().sort_values(CLAVE).reset_index(drop=True)

n_ambos = len(p_focos.merge(p_deter, on=CLAVE))
print(f"Municipios en focos: {len(p_focos)} | en DETER: {len(p_deter)} | en ambos: {n_ambos} | universo (union): {len(universo)}")

# Municipios con alertas DETER pero sin ningun foco no tienen centroide (los centroides salen de focos).
# SIN_CENTROIDE decide que hacer: "detener" (por defecto, el notebook se detiene y los lista)
# o "excluir" (se quitan del panel junto con sus alertas DETER). Es una decision del equipo.
SIN_CENTROIDE = "excluir"
sin_cent = universo.merge(centroides[CLAVE], on=CLAVE, how="left", indicator=True).query("_merge == 'left_only'")[CLAVE]
excluidas_deter = 0
if len(sin_cent):
    print(f"Municipios sin centroide ({len(sin_cent)}):", sorted(map(tuple, sin_cent.values)))
    if SIN_CENTROIDE == "excluir":
        n_antes = len(deter_ok)
        universo = universo.merge(centroides[CLAVE], on=CLAVE, how="inner")
        deter_ok = deter_ok.merge(universo[CLAVE], on=CLAVE, how="inner")
        excluidas_deter = n_antes - len(deter_ok)
        print(f"Excluidos del panel. Alertas DETER excluidas con ellos: {excluidas_deter:,}")
    else:
        raise AssertionError("Hay municipios con alertas DETER pero sin focos, y por eso sin centroide. "
                             "Decidir con el equipo: SIN_CENTROIDE = 'excluir' o dar otra fuente de centroide.")

grilla = universo.merge(pd.DataFrame({"semana": SEMANAS}), how="cross")
print("Grilla:", grilla.shape, f"= {len(universo)} municipios x {len(SEMANAS)} semanas")

Municipios en focos: 543 | en DETER: 480 | en ambos: 471 | universo (union): 552
Municipios sin centroide (9): [('AUGUSTINOPOLIS', 'TO'), ('BARAO DE MELGACO', 'MT'), ('CAMPO NOVO DO PARECIS', 'MT'), ('NOSSA SENHORA DO LIVRAMENTO', 'MT'), ('NOVO SANTO ANTONIO', 'MT'), ('SAMPAIO', 'TO'), ('SANTO ANTONIO DO LEVERGER', 'MT'), ('SAPEZAL', 'MT'), ('SERRA NOVA DOURADA', 'MT')]
Excluidos del panel. Alertas DETER excluidas con ellos: 242
Grilla: (136293, 3) = 543 municipios x 251 semanas


### Hallazgos de esta etapa

**Qué se observa**
- El universo tiene **552** municipios (unión de focos y DETER). Solo **543** tienen centroide, porque tienen focos.
- Se excluyeron **9**: `AUGUSTINOPOLIS` (TO), `BARAO DE MELGACO` (MT), `CAMPO NOVO DO PARECIS` (MT), `NOSSA SENHORA DO LIVRAMENTO` (MT), `NOVO SANTO ANTONIO` (MT), `SAMPAIO` (TO), `SANTO ANTONIO DO LEVERGER` (MT), `SAPEZAL` (MT) y `SERRA NOVA DOURADA` (MT). Con ellos se perdieron **242 alertas DETER**.
- La grilla tiene **543 municipios x 251 semanas = 136,293 filas**. Quedaron fuera del calendario **3,044 focos** y **601 alertas DETER**.

**Por qué se decidió así**
- BDQueimadas solo registra los días con foco. La grilla completa agrega las semanas sin focos como **ceros reales**; sin ella el modelo solo vería semanas con fuego.
- La semana va de lunes a domingo y se identifica por su lunes, para que todas las fuentes (focos, DETER, SISAM, NASA POWER) usen el mismo corte.
- Sin centroide no hay forma de consultar NASA POWER ni de ubicar el municipio en un mapa.

**Qué implica para el modelado**
- **Limitación:** el panel cubre solo los municipios con focos del bioma Amazônia. Los 9 municipios excluidos y sus 242 alertas DETER no están en el dataset.


## 3. Agregado semanal de focos

Por `(semana, municipio_norm, uf)`: número de focos, FRP total y promedio, precipitación y días sin lluvia promedio (valores que reporta BDQueimadas en cada foco), y promedio y máximo de `RiscoFogo`. En la grilla, `num_incendios` y `frp_total` valen 0 cuando no hubo focos (cero real). Las demás variables quedan en **NaN** sin focos: no se imputan aquí.

In [5]:
focos_semanal = (
    focos_ok
    .groupby(["semana"] + CLAVE)
    .agg(
        num_incendios=("FRP", "size"),
        frp_total=("FRP", "sum"),
        frp_promedio=("FRP", "mean"),
        precipitacion_promedio=("Precipitacao", "mean"),
        dias_sin_lluvia_promedio=("DiaSemChuva", "mean"),
        riesgo_fuego_promedio=("RiscoFogo", "mean"),
        riesgo_fuego_max=("RiscoFogo", "max"),
    )
    .reset_index()
)
print("Semanas-municipio con focos:", focos_semanal.shape)

panel = grilla.merge(focos_semanal, on=["semana"] + CLAVE, how="left", validate="one_to_one")
panel[["num_incendios", "frp_total"]] = panel[["num_incendios", "frp_total"]].fillna(0)
panel["num_incendios"] = panel["num_incendios"].astype(int)
print("Panel con focos:", panel.shape)

Semanas-municipio con focos: (39162, 10)
Panel con focos: (136293, 10)


## 4. Contexto DETER (para el KPI)

Alertas, hectáreas y alertas dentro de área protegida por semana y municipio, con 0 donde no hubo alertas. **DETER no se usa como target ni como predictora.**

In [6]:
deter_semanal = (
    deter_ok
    .groupby(["semana"] + CLAVE)
    .agg(
        deter_num_alertas=("area_ha", "size"),
        deter_area_ha=("area_ha", "sum"),
        deter_num_alertas_area_protegida=("dentro_area_protegida", "sum"),
    )
    .reset_index()
)

panel = panel.merge(deter_semanal, on=["semana"] + CLAVE, how="left", validate="one_to_one")
cols_deter = ["deter_num_alertas", "deter_area_ha", "deter_num_alertas_area_protegida"]
panel[cols_deter] = panel[cols_deter].fillna(0)
panel["deter_num_alertas"] = panel["deter_num_alertas"].astype(int)
panel["deter_num_alertas_area_protegida"] = panel["deter_num_alertas_area_protegida"].astype(int)
print(panel.shape)

(136293, 13)


## 5. Centroides

`lat` y `lon` salen de `centroides_municipio.csv` (promedio de los focos de 2022 a 2024, o de 2025 y 2026 si el municipio no tuvo focos antes). Un municipio con alertas DETER pero sin ningún foco no tendría centroide: en ese caso el notebook se detiene y lista los municipios.

In [7]:
panel = panel.merge(centroides[CLAVE + ["lat", "lon"]], on=CLAVE, how="left", validate="many_to_one")

sin_centroide = panel.loc[panel["lat"].isna() | panel["lon"].isna(), CLAVE].drop_duplicates()
if len(sin_centroide):
    print("Municipios sin centroide (solo tienen alertas DETER, sin focos):")
    print(sin_centroide.to_string(index=False))

orden = ["semana", "municipio_norm", "uf", "lat", "lon", "num_incendios", "frp_total", "frp_promedio",
         "precipitacion_promedio", "dias_sin_lluvia_promedio", "riesgo_fuego_promedio", "riesgo_fuego_max",
         "deter_num_alertas", "deter_area_ha", "deter_num_alertas_area_protegida"]
panel = panel[orden].sort_values(CLAVE + ["semana"]).reset_index(drop=True)

## 6. Controles

Se verifica con `assert`: clave sin duplicados, filas = municipios por semanas, conservación de focos y hectáreas DETER dentro del calendario, ningún valor negativo y `lat`/`lon` sin nulos.

In [8]:
assert not panel.duplicated(CLAVE + ["semana"]).any(), "Clave (municipio_norm, uf, semana) duplicada"
assert len(panel) == len(universo) * len(SEMANAS), "Filas del panel != municipios x semanas"
assert set(panel["semana"].unique()) == set(SEMANAS), "Las semanas del panel no coinciden con el calendario"

assert panel["num_incendios"].sum() == len(focos_ok), \
    f"Focos del panel ({panel['num_incendios'].sum():,}) != focos del calendario ({len(focos_ok):,})"
assert np.isclose(panel["deter_area_ha"].sum(), deter_ok["area_ha"].sum()), "Hectareas DETER del panel != DETER del calendario"
assert panel["deter_num_alertas"].sum() == len(deter_ok), "Alertas DETER del panel != DETER del calendario"

num_cols = panel.select_dtypes("number").columns.drop(["lat", "lon"])
assert (panel[num_cols].fillna(0) >= 0).all().all(), "Hay valores negativos"
assert panel["lat"].notna().all() and panel["lon"].notna().all(), "Hay municipios sin lat/lon"

print("Todos los controles pasaron.")
print(f"Panel: {len(panel):,} filas = {len(universo)} municipios x {len(SEMANAS)} semanas")
print(f"Focos conservados: {panel['num_incendios'].sum():,} | descartados fuera del calendario: {descartados_focos:,}")
print(f"Alertas DETER conservadas: {panel['deter_num_alertas'].sum():,} | descartadas fuera del calendario: {descartados_deter:,} | excluidas por municipio sin centroide: {excluidas_deter:,}")
prop = (panel["num_incendios"] > 0).mean() * 100
print(f"\nProporcion de semanas-municipio con focos: {prop:.1f}% (referencia Hito 2, 2022 a 2024: ~32%)")
print((panel.assign(anio=panel["semana"].dt.year).groupby("anio")["num_incendios"].apply(lambda s: (s > 0).mean() * 100)).round(1))

Todos los controles pasaron.
Panel: 136,293 filas = 543 municipios x 251 semanas
Focos conservados: 442,592 | descartados fuera del calendario: 3,044
Alertas DETER conservadas: 220,837 | descartadas fuera del calendario: 601 | excluidas por municipio sin centroide: 242

Proporcion de semanas-municipio con focos: 28.7% (referencia Hito 2, 2022 a 2024: ~32%)
anio
2022    29.5
2023    32.9
2024    33.3
2025    24.7
2026    21.9
Name: num_incendios, dtype: float64


### Hallazgos de esta etapa

**Qué se observa**
- Todos los controles pasaron. Se conservaron **442,592** focos.
- El **28.7 %** de las semanas-municipio tiene focos (referencia del Hito 2: cerca de 32 %). Por año: **29.5 %** (2022), **32.9 %** (2023), **33.3 %** (2024), **24.7 %** (2025) y **21.9 %** (2026).

**Por qué se decidió así**
- Los ceros se conservan porque son semanas reales sin focos.

**Qué implica para el modelado**
- Cerca del **71.3 %** de las filas tiene 0 focos: el problema está desbalanceado.
- **Limitación:** el porcentaje cae en 2025 y 2026. Un periodo de prueba en esos años tendrá menos semanas con focos que el entrenamiento.


## 7. Comparación informativa con el panel del Hito 2

Sin `assert`. En las filas cuya clave `(semana, municipio_norm, uf)` coincide con `dataset_semanal_COMPLETO.csv` (Hito 2), `num_incendios` debe ser igual. Las hectáreas DETER pueden diferir porque la extracción de DETER cambió (snapshot nuevo).

In [9]:
if not RUTA_PANEL_HITO2.exists():
    print("No existe dataset_semanal_COMPLETO.csv: se omite la comparacion.")
else:
    viejo = pd.read_csv(RUTA_PANEL_HITO2, parse_dates=["semana"], usecols=["semana", "municipio_norm", "uf", "num_incendios", "area_ha_total"])
    comp = viejo.merge(panel[["semana"] + CLAVE + ["num_incendios", "deter_area_ha"]], on=["semana"] + CLAVE,
                       how="inner", suffixes=("_hito2", "_nuevo"))
    print(f"Filas del Hito 2: {len(viejo):,} | con clave coincidente: {len(comp):,}")
    dif_f = (comp["num_incendios_hito2"] != comp["num_incendios_nuevo"]).sum()
    print(f"Filas con num_incendios distinto: {dif_f:,}")
    dif_ha = comp["deter_area_ha"] - comp["area_ha_total"]
    print(f"Hectareas DETER: total Hito 2 {comp['area_ha_total'].sum():,.0f} | total nuevo {comp['deter_area_ha'].sum():,.0f}")
    print(f"Filas con diferencia de hectareas > 0.01: {(dif_ha.abs() > 0.01).sum():,}")
    print("(las diferencias de hectareas son esperables por el cambio de snapshot de DETER)")

Filas del Hito 2: 11,445 | con clave coincidente: 11,410
Filas con num_incendios distinto: 0
Hectareas DETER: total Hito 2 3,332,395 | total nuevo 10,381,049
Filas con diferencia de hectareas > 0.01: 9,705
(las diferencias de hectareas son esperables por el cambio de snapshot de DETER)


### Hallazgos de esta etapa

**Qué se observa**
- De **11,445** filas del panel del Hito 2, **11,410** tienen clave coincidente. En ellas hay **0** filas con `num_incendios` distinto.
- Las hectáreas DETER suman **3,332,395** en el Hito 2 y **10,381,049** en el panel nuevo. **9,705** filas difieren en más de 0.01 ha.

**Por qué se decidió así**
- La comparación es solo informativa (no hay `assert`). Sirve para confirmar que el cambio de calendario y de grilla no alteró los conteos de focos.
- Las hectáreas cambian porque DETER se extrajo de nuevo completo (ver notebook 01).


In [10]:
panel.to_csv(RUTA_SALIDA, index=False, encoding="utf-8")
print("Guardado:", RUTA_SALIDA.relative_to(BASE), panel.shape)
panel.head()

Guardado: datos/datasets_preprocesamiento/panel_semanal_base.csv (136293, 15)


,semana,municipio_norm,uf,lat,lon,num_incendios,frp_total,frp_promedio,precipitacion_promedio,dias_sin_lluvia_promedio,riesgo_fuego_promedio,riesgo_fuego_max,deter_num_alertas,deter_area_ha,deter_num_alertas_area_protegida
0,2022-01-03,ABAETETUBA,PA,-1.773752,-48.785494,0,0.0,NaN,NaN,NaN,NaN,NaN,0,0.0,0
1,2022-01-10,ABAETETUBA,PA,-1.773752,-48.785494,0,0.0,NaN,NaN,NaN,NaN,NaN,0,0.0,0
2,2022-01-17,ABAETETUBA,PA,-1.773752,-48.785494,0,0.0,NaN,NaN,NaN,NaN,NaN,0,0.0,0
3,2022-01-24,ABAETETUBA,PA,-1.773752,-48.785494,0,0.0,NaN,NaN,NaN,NaN,NaN,0,0.0,0
4,2022-01-31,ABAETETUBA,PA,-1.773752,-48.785494,0,0.0,NaN,NaN,NaN,NaN,NaN,0,0.0,0
